# Time-Series Forecasting: Nicotine Craving and Anxiety Dynamics

This notebook presents a complete, production-grade machine learning pipeline for forecasting **nicotine craving intensity** and **daily anxiety scores** over a 30-day monitoring window. The dataset contains longitudinal records from 1,000 participants, each observed across 30 consecutive days, yielding 30,000 rows of richly annotated behavioral and physiological data.

The pipeline covers the following stages:

1. Library imports and environment configuration
2. Dataset loading and initial inspection
3. Exploratory data analysis (EDA)
4. Missing value treatment and data cleaning
5. Feature engineering (lag, rolling-window, interaction features)
6. Temporal train / validation / test split
7. Preprocessing pipeline construction
8. Baseline model
9. Linear regularized regression models
10. Ensemble tree-based models
11. XGBoost and LightGBM models
12. LSTM deep learning model
13. Model comparison and evaluation summary
14. Feature importance analysis
15. Residual analysis
16. Forecast visualization
17. Per-participant forecast
18. Cross-validation with TimeSeriesSplit
19. Artifact export and experiment summary

## 1. Library Imports and Environment Configuration

All required libraries are imported here. Random seeds for NumPy, Python built-in random, and TensorFlow are fixed to ensure full reproducibility across runs.

In [ ]:
import os
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json

from pathlib import Path
from datetime import datetime

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, TimeSeriesSplit
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    mean_absolute_percentage_error
)
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor
)

import xgboost as xgb
import lightgbm as lgb

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('muted')

print(f'NumPy   : {np.__version__}')
print(f'Pandas  : {pd.__version__}')
print(f'XGBoost : {xgb.__version__}')
print(f'LightGBM: {lgb.__version__}')
print(f'TF      : {tf.__version__}')

## 2. Configuration and Path Setup

Centralized path management ensures that all artifacts (trained models, scalers, evaluation metrics, and plots) are written to `Model/artifacts/`, making the project fully portable and reproducible.

In [ ]:
BASE_DIR     = Path('.').resolve().parent
DATA_PATH    = BASE_DIR / 'Dataset' / 'Nicotine_Craving_Anxiety_Dynamics_30Days.csv'
ARTIFACT_DIR = Path('.') / 'artifacts'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_CRAVING = 'Overall_Craving_Score'
TARGET_ANXIETY = 'Daily_Anxiety_Score'
TIME_COL       = 'Day_Number'
GROUP_COL      = 'Participant_ID'

print(f'Data path   : {DATA_PATH}')
print(f'Artifact dir: {ARTIFACT_DIR.resolve()}')

## 3. Dataset Loading and Initial Inspection

The dataset is loaded and an initial structural audit is performed. This includes checking dimensionality, data types, and the first few rows to validate that the load was successful.

In [ ]:
df_raw = pd.read_csv(DATA_PATH)

print(f'Shape: {df_raw.shape}')
print(f'Columns ({len(df_raw.columns)}): {list(df_raw.columns)}')
df_raw.head()

In [ ]:
df_raw.dtypes

In [ ]:
df_raw.describe(include='all').T

## 4. Exploratory Data Analysis (EDA)

EDA is performed across three dimensions:

- **Missing value analysis**: Quantifying the extent and patterns of missingness.
- **Target distribution analysis**: Understanding the statistical properties of the two forecast targets.
- **Temporal trend analysis**: Visualizing how craving and anxiety scores evolve across the 30-day window.

In [ ]:
missing     = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_df  = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df  = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)
print('Columns with missing values:')
print(missing_df.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_raw[TARGET_CRAVING].dropna(), bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of Overall Craving Score')
axes[0].set_xlabel(TARGET_CRAVING)
axes[0].set_ylabel('Frequency')

axes[1].hist(df_raw[TARGET_ANXIETY].dropna(), bins=40, color='coral', edgecolor='white')
axes[1].set_title('Distribution of Daily Anxiety Score')
axes[1].set_xlabel(TARGET_ANXIETY)
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'target_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
daily_avg = df_raw.groupby(TIME_COL)[[TARGET_CRAVING, TARGET_ANXIETY]].mean().reset_index()

fig, ax1 = plt.subplots(figsize=(14, 5))
ax2 = ax1.twinx()

ax1.plot(daily_avg[TIME_COL], daily_avg[TARGET_CRAVING], 'o-', color='steelblue', label='Craving Score')
ax2.plot(daily_avg[TIME_COL], daily_avg[TARGET_ANXIETY], 's--', color='coral', label='Anxiety Score')

ax1.set_xlabel('Day Number')
ax1.set_ylabel('Average Craving Score', color='steelblue')
ax2.set_ylabel('Average Anxiety Score', color='coral')
ax1.set_title('Population-level Average Craving and Anxiety Over 30 Days')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

plt.savefig(ARTIFACT_DIR / 'temporal_trends.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
numeric_cols = df_raw.select_dtypes(include=np.number).columns.tolist()
corr_matrix  = df_raw[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(18, 14))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=False, cmap='RdBu_r', center=0, ax=ax, linewidths=0.3)
ax.set_title('Feature Correlation Matrix (Lower Triangle)')
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
sample_ids = df_raw[GROUP_COL].unique()[:6]
fig, axes  = plt.subplots(2, 3, figsize=(16, 8))

for ax, pid in zip(axes.flatten(), sample_ids):
    sub = df_raw[df_raw[GROUP_COL] == pid].sort_values(TIME_COL)
    ax.plot(sub[TIME_COL], sub[TARGET_CRAVING], 'o-', label='Craving', color='steelblue', markersize=4)
    ax.plot(sub[TIME_COL], sub[TARGET_ANXIETY],  's--', label='Anxiety',  color='coral',     markersize=4)
    ax.set_title(f'Participant {pid}')
    ax.set_xlabel('Day')
    ax.set_ylabel('Score')
    ax.legend(fontsize=8)

plt.suptitle('Individual Craving and Anxiety Trajectories (6 Sample Participants)', fontsize=13)
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'individual_trajectories.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Data Cleaning and Preprocessing

The cleaning stage addresses:

- **Sorting**: Records are sorted by participant and day to preserve temporal ordering.
- **Categorical encoding**: String columns (gender, occupation, triggers) are ordinally encoded.
- **Time-based parsing**: The `First_Pouch_Time` column is converted to minutes since midnight.
- **Binary encoding**: Yes/No columns are mapped to 1/0 integers.

In [ ]:
df = df_raw.copy()
df = df.sort_values([GROUP_COL, TIME_COL]).reset_index(drop=True)

def parse_time_to_minutes(t):
    if pd.isna(t):
        return np.nan
    try:
        parts = str(t).split(':')
        return int(parts[0]) * 60 + int(parts[1])
    except Exception:
        return np.nan

df['First_Pouch_Minutes'] = df['First_Pouch_Time'].apply(parse_time_to_minutes)
df.drop(columns=['First_Pouch_Time'], inplace=True)

binary_cols = ['Attempted_To_Resist_Craving', 'Successfully_Resisted', 'Major_Stress_Event_Today']
for col in binary_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0})

categorical_cols = df.select_dtypes(include='object').columns.tolist()
categorical_cols = [c for c in categorical_cols if c != GROUP_COL]

label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

print(f'Remaining object columns: {df.select_dtypes(include="object").columns.tolist()}')
print(f'Shape after encoding: {df.shape}')

## 6. Feature Engineering

Time-series forecasting models benefit greatly from features that capture temporal dynamics. The following feature groups are constructed:

- **Lag features**: Previous-day values of the target variables, providing direct autoregressive signals.
- **Rolling statistics**: 3-day and 7-day rolling mean and standard deviation, capturing short- and medium-term trends.
- **Temporal ratio features**: Ratios between morning and evening sub-scores to capture diurnal asymmetry.
- **Interaction features**: Compound features encoding known clinical relationships, such as craving intensity multiplied by perceived stress.

In [ ]:
LAG_PERIODS  = [1, 2, 3]
ROLL_WINDOWS = [3, 7]

def build_lag_rolling_features(df, target_col, group_col, lags, windows):
    g = df.groupby(group_col)[target_col]
    for lag in lags:
        df[f'{target_col}_lag{lag}'] = g.shift(lag)
    for w in windows:
        df[f'{target_col}_roll_mean{w}'] = g.shift(1).transform(
            lambda x: x.rolling(w, min_periods=1).mean()
        )
        df[f'{target_col}_roll_std{w}'] = g.shift(1).transform(
            lambda x: x.rolling(w, min_periods=1).std()
        )
    return df

df = build_lag_rolling_features(df, TARGET_CRAVING, GROUP_COL, LAG_PERIODS, ROLL_WINDOWS)
df = build_lag_rolling_features(df, TARGET_ANXIETY,  GROUP_COL, LAG_PERIODS, ROLL_WINDOWS)

if 'Morning_Cravings_Score' in df.columns and 'Evening_Cravings_Score' in df.columns:
    df['Craving_Diurnal_Ratio'] = df['Morning_Cravings_Score'] / (df['Evening_Cravings_Score'] + 1e-6)

if 'Morning_Anxiety_Score' in df.columns and 'Evening_Anxiety_Score' in df.columns:
    df['Anxiety_Diurnal_Ratio'] = df['Morning_Anxiety_Score'] / (df['Evening_Anxiety_Score'] + 1e-6)

if 'Perceived_Stress_Score' in df.columns and 'Craving_Intensity_Peak' in df.columns:
    df['Stress_x_Craving_Peak'] = df['Perceived_Stress_Score'] * df['Craving_Intensity_Peak']

if 'Sleep_Quality_Score' in df.columns and 'Sleep_Duration_Hours' in df.columns:
    df['Sleep_Composite'] = df['Sleep_Quality_Score'] * df['Sleep_Duration_Hours']

print(f'Shape after feature engineering: {df.shape}')
new_feats = [c for c in df.columns if any(k in c for k in ['lag', 'roll', 'Ratio', 'Composite', 'x_'])]
print(f'Engineered features ({len(new_feats)}): {new_feats}')

## 7. Temporal Train / Validation / Test Split

To respect the temporal ordering of observations and prevent data leakage, the dataset is split by `Day_Number` rather than by random row sampling:

- **Training set**: Days 1 through 20 (67% of the timeline)
- **Validation set**: Days 21 through 25 (17% of the timeline)
- **Test set**: Days 26 through 30 (final hold-out, 17% of the timeline)

Rows where the target variable is missing are excluded from each split.

In [ ]:
TRAIN_END = 20
VAL_END   = 25

train_mask = df[TIME_COL] <= TRAIN_END
val_mask   = (df[TIME_COL] > TRAIN_END) & (df[TIME_COL] <= VAL_END)
test_mask  = df[TIME_COL] > VAL_END

print(f'Training rows   : {train_mask.sum():,}')
print(f'Validation rows : {val_mask.sum():,}')
print(f'Test rows       : {test_mask.sum():,}')

In [ ]:
NON_FEATURE_COLS = [GROUP_COL, TIME_COL, TARGET_CRAVING, TARGET_ANXIETY]
FEATURE_COLS     = [c for c in df.columns if c not in NON_FEATURE_COLS]

def get_split(df, mask, target_col, feature_cols):
    sub = df[mask].dropna(subset=[target_col]).copy()
    return sub[feature_cols], sub[target_col]

X_train_c, y_train_c = get_split(df, train_mask, TARGET_CRAVING, FEATURE_COLS)
X_val_c,   y_val_c   = get_split(df, val_mask,   TARGET_CRAVING, FEATURE_COLS)
X_test_c,  y_test_c  = get_split(df, test_mask,  TARGET_CRAVING, FEATURE_COLS)

X_train_a, y_train_a = get_split(df, train_mask, TARGET_ANXIETY, FEATURE_COLS)
X_val_a,   y_val_a   = get_split(df, val_mask,   TARGET_ANXIETY, FEATURE_COLS)
X_test_a,  y_test_a  = get_split(df, test_mask,  TARGET_ANXIETY, FEATURE_COLS)

print(f'Craving - Train: {X_train_c.shape}, Val: {X_val_c.shape}, Test: {X_test_c.shape}')
print(f'Anxiety - Train: {X_train_a.shape}, Val: {X_val_a.shape}, Test: {X_test_a.shape}')
print(f'Total features : {len(FEATURE_COLS)}')

## 8. Preprocessing Pipeline

A `sklearn` `Pipeline` is constructed to handle imputation and scaling in a principled, leak-free manner. The pipeline is fit exclusively on the training set and applied identically to validation and test sets, ensuring no information from future folds contaminates the learned transformations.

In [ ]:
preprocessor = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

X_train_c_pp = preprocessor.fit_transform(X_train_c)
X_val_c_pp   = preprocessor.transform(X_val_c)
X_test_c_pp  = preprocessor.transform(X_test_c)

preprocessor_a = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

X_train_a_pp = preprocessor_a.fit_transform(X_train_a)
X_val_a_pp   = preprocessor_a.transform(X_val_a)
X_test_a_pp  = preprocessor_a.transform(X_test_a)

joblib.dump(preprocessor,   ARTIFACT_DIR / 'preprocessor_craving.pkl')
joblib.dump(preprocessor_a, ARTIFACT_DIR / 'preprocessor_anxiety.pkl')
print('Preprocessors fitted and saved.')

## 9. Evaluation Utility

A unified evaluation function computes four regression metrics:

- **MAE** (Mean Absolute Error): Average absolute deviation in original score units.
- **RMSE** (Root Mean Squared Error): Penalizes large errors more heavily than MAE.
- **MAPE** (Mean Absolute Percentage Error): Scale-independent error expressed as a percentage.
- **R2** (Coefficient of Determination): Proportion of variance in the target explained by the model.

In [ ]:
def evaluate_model(y_true, y_pred, model_name='Model', split_name='Test'):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    r2   = r2_score(y_true, y_pred)
    return {
        'Model': model_name,
        'Split': split_name,
        'MAE':   round(mae,  4),
        'RMSE':  round(rmse, 4),
        'MAPE':  round(mape, 4),
        'R2':    round(r2,   4)
    }

results_craving = []
results_anxiety = []
print('Evaluation function defined.')

## 10. Baseline Model

Before training complex models, a naive persistence baseline is established. The persistence model predicts that tomorrow's score equals today's score (lag-1 as the sole prediction). Any learning model should comfortably exceed this benchmark.

In [ ]:
craving_lag1_col = f'{TARGET_CRAVING}_lag1'
anxiety_lag1_col = f'{TARGET_ANXIETY}_lag1'

if craving_lag1_col in FEATURE_COLS:
    test_df_c    = df[test_mask].dropna(subset=[TARGET_CRAVING]).copy()
    lag_preds_c  = test_df_c[craving_lag1_col].fillna(test_df_c[TARGET_CRAVING].mean())
    m = evaluate_model(y_test_c, lag_preds_c.values, 'Persistence Baseline', 'Test')
    results_craving.append(m)
    print(f'[Craving] Persistence Baseline: {m}')

if anxiety_lag1_col in FEATURE_COLS:
    test_df_a    = df[test_mask].dropna(subset=[TARGET_ANXIETY]).copy()
    lag_preds_a  = test_df_a[anxiety_lag1_col].fillna(test_df_a[TARGET_ANXIETY].mean())
    m = evaluate_model(y_test_a, lag_preds_a.values, 'Persistence Baseline', 'Test')
    results_anxiety.append(m)
    print(f'[Anxiety] Persistence Baseline: {m}')

## 11. Linear Regularized Regression Models

Ridge and Lasso regression models are trained as interpretable linear baselines. Ridge applies L2 regularization, which smooths the coefficient distribution. Lasso applies L1 regularization, which induces sparsity and can perform implicit feature selection. ElasticNet combines both penalties.

In [ ]:
linear_configs = {
    'Ridge':      Ridge(alpha=1.0, random_state=SEED),
    'Lasso':      Lasso(alpha=0.01, random_state=SEED, max_iter=5000),
    'ElasticNet': ElasticNet(alpha=0.01, l1_ratio=0.5, random_state=SEED, max_iter=5000)
}

for name, model in linear_configs.items():
    model.fit(X_train_c_pp, y_train_c)
    preds = model.predict(X_test_c_pp)
    m = evaluate_model(y_test_c, preds, name, 'Test')
    results_craving.append(m)
    print(f'[Craving] {name}: MAE={m["MAE"]:.4f}, RMSE={m["RMSE"]:.4f}, R2={m["R2"]:.4f}')
    joblib.dump(model, ARTIFACT_DIR / f'craving_{name.lower()}.pkl')

print()

for name, model in linear_configs.items():
    model.fit(X_train_a_pp, y_train_a)
    preds = model.predict(X_test_a_pp)
    m = evaluate_model(y_test_a, preds, name, 'Test')
    results_anxiety.append(m)
    print(f'[Anxiety] {name}: MAE={m["MAE"]:.4f}, RMSE={m["RMSE"]:.4f}, R2={m["R2"]:.4f}')

## 12. Ensemble Tree-Based Models

Three ensemble methods are evaluated:

- **Random Forest**: Bagging of decorrelated decision trees; robust to noise and overfitting.
- **Gradient Boosting (sklearn)**: Sequential boosting with shrinkage; strong out-of-the-box performance.
- **Histogram Gradient Boosting**: Binning-based boosting that natively handles missing values and scales to large datasets.

In [ ]:
ensemble_configs = {
    'RandomForest': RandomForestRegressor(
        n_estimators=200, max_depth=10, min_samples_leaf=5, n_jobs=-1, random_state=SEED
    ),
    'GradientBoosting': GradientBoostingRegressor(
        n_estimators=200, learning_rate=0.05, max_depth=5, subsample=0.8, random_state=SEED
    ),
    'HistGradientBoosting': HistGradientBoostingRegressor(
        max_iter=300, learning_rate=0.05, max_depth=6, random_state=SEED
    )
}

for name, model in ensemble_configs.items():
    model.fit(X_train_c_pp, y_train_c)
    preds = model.predict(X_test_c_pp)
    m = evaluate_model(y_test_c, preds, name, 'Test')
    results_craving.append(m)
    print(f'[Craving] {name}: MAE={m["MAE"]:.4f}, RMSE={m["RMSE"]:.4f}, R2={m["R2"]:.4f}')
    joblib.dump(model, ARTIFACT_DIR / f'craving_{name.lower()}.pkl')

print()

for name, model in ensemble_configs.items():
    model.fit(X_train_a_pp, y_train_a)
    preds = model.predict(X_test_a_pp)
    m = evaluate_model(y_test_a, preds, name, 'Test')
    results_anxiety.append(m)
    print(f'[Anxiety] {name}: MAE={m["MAE"]:.4f}, RMSE={m["RMSE"]:.4f}, R2={m["R2"]:.4f}')

## 13. XGBoost and LightGBM Models

XGBoost and LightGBM are industry-standard gradient boosting frameworks that consistently win tabular forecasting competitions. Key algorithmic differences:

- **XGBoost**: Exact greedy split finding with second-order Taylor expansion of the loss.
- **LightGBM**: Leaf-wise growth strategy with gradient-based one-side sampling; significantly faster on large datasets.

Early stopping on the validation set is used to prevent overfitting and determine the optimal number of boosting rounds.

In [ ]:
xgb_params = dict(
    n_estimators=1000, learning_rate=0.03, max_depth=6,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0,
    random_state=SEED, n_jobs=-1
)

xgb_c = xgb.XGBRegressor(**xgb_params, early_stopping_rounds=50, eval_metric='rmse')
xgb_c.fit(X_train_c_pp, y_train_c, eval_set=[(X_val_c_pp, y_val_c)], verbose=False)
preds = xgb_c.predict(X_test_c_pp)
m = evaluate_model(y_test_c, preds, 'XGBoost', 'Test')
results_craving.append(m)
print(f'[Craving] XGBoost: MAE={m["MAE"]:.4f}, RMSE={m["RMSE"]:.4f}, R2={m["R2"]:.4f}')
xgb_c.save_model(str(ARTIFACT_DIR / 'craving_xgboost.json'))

xgb_a = xgb.XGBRegressor(**xgb_params, early_stopping_rounds=50, eval_metric='rmse')
xgb_a.fit(X_train_a_pp, y_train_a, eval_set=[(X_val_a_pp, y_val_a)], verbose=False)
preds = xgb_a.predict(X_test_a_pp)
m = evaluate_model(y_test_a, preds, 'XGBoost', 'Test')
results_anxiety.append(m)
print(f'[Anxiety] XGBoost: MAE={m["MAE"]:.4f}, RMSE={m["RMSE"]:.4f}, R2={m["R2"]:.4f}')
xgb_a.save_model(str(ARTIFACT_DIR / 'anxiety_xgboost.json'))

In [ ]:
lgb_params = dict(
    n_estimators=1000, learning_rate=0.03, max_depth=6,
    num_leaves=63, subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0,
    random_state=SEED, n_jobs=-1, verbose=-1
)

lgb_c = lgb.LGBMRegressor(**lgb_params)
lgb_c.fit(
    X_train_c_pp, y_train_c,
    eval_set=[(X_val_c_pp, y_val_c)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(period=-1)]
)
preds = lgb_c.predict(X_test_c_pp)
m = evaluate_model(y_test_c, preds, 'LightGBM', 'Test')
results_craving.append(m)
print(f'[Craving] LightGBM: MAE={m["MAE"]:.4f}, RMSE={m["RMSE"]:.4f}, R2={m["R2"]:.4f}')
lgb_c.booster_.save_model(str(ARTIFACT_DIR / 'craving_lightgbm.txt'))

lgb_a = lgb.LGBMRegressor(**lgb_params)
lgb_a.fit(
    X_train_a_pp, y_train_a,
    eval_set=[(X_val_a_pp, y_val_a)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(period=-1)]
)
preds = lgb_a.predict(X_test_a_pp)
m = evaluate_model(y_test_a, preds, 'LightGBM', 'Test')
results_anxiety.append(m)
print(f'[Anxiety] LightGBM: MAE={m["MAE"]:.4f}, RMSE={m["RMSE"]:.4f}, R2={m["R2"]:.4f}')
lgb_a.booster_.save_model(str(ARTIFACT_DIR / 'anxiety_lightgbm.txt'))

## 14. LSTM Deep Learning Model

Long Short-Term Memory (LSTM) networks are a class of recurrent neural networks specifically designed to learn long-range temporal dependencies. The architecture used here:

- The tabular data is reshaped into a 3D tensor `(samples, timesteps=1, features)` to interface with the LSTM layer.
- Two stacked LSTM layers with dropout regularization capture both local and global temporal patterns.
- A dense output head produces the final scalar prediction.
- `ReduceLROnPlateau` dynamically reduces the learning rate when validation loss stagnates.
- `EarlyStopping` with best-weight restoration halts training at the optimal point.

In [ ]:
def build_lstm_model(input_dim, units=64, dropout=0.3, lr=1e-3):
    inp = keras.Input(shape=(1, input_dim))
    x   = layers.LSTM(units, return_sequences=True, dropout=dropout)(inp)
    x   = layers.LSTM(units // 2, dropout=dropout)(x)
    x   = layers.Dense(32, activation='relu')(x)
    x   = layers.Dropout(dropout)(x)
    out = layers.Dense(1)(x)
    model = keras.Model(inp, out)
    model.compile(optimizer=keras.optimizers.Adam(lr), loss='mse', metrics=['mae'])
    return model

n_features = X_train_c_pp.shape[1]
lstm_c = build_lstm_model(n_features)
lstm_c.summary()

In [ ]:
X_train_3d_c = X_train_c_pp.reshape(-1, 1, n_features)
X_val_3d_c   = X_val_c_pp.reshape(-1,   1, n_features)
X_test_3d_c  = X_test_c_pp.reshape(-1,  1, n_features)

cb_list = [
    callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, min_lr=1e-6)
]

history_c = lstm_c.fit(
    X_train_3d_c, y_train_c.values,
    validation_data=(X_val_3d_c, y_val_c.values),
    epochs=200, batch_size=256, callbacks=cb_list, verbose=0
)

preds = lstm_c.predict(X_test_3d_c, verbose=0).flatten()
m = evaluate_model(y_test_c, preds, 'LSTM', 'Test')
results_craving.append(m)
print(f'[Craving] LSTM: MAE={m["MAE"]:.4f}, RMSE={m["RMSE"]:.4f}, R2={m["R2"]:.4f}')
lstm_c.save(str(ARTIFACT_DIR / 'craving_lstm.keras'))

n_features_a = X_train_a_pp.shape[1]
X_train_3d_a = X_train_a_pp.reshape(-1, 1, n_features_a)
X_val_3d_a   = X_val_a_pp.reshape(-1,   1, n_features_a)
X_test_3d_a  = X_test_a_pp.reshape(-1,  1, n_features_a)

lstm_a = build_lstm_model(n_features_a)
history_a = lstm_a.fit(
    X_train_3d_a, y_train_a.values,
    validation_data=(X_val_3d_a, y_val_a.values),
    epochs=200, batch_size=256, callbacks=cb_list, verbose=0
)

preds = lstm_a.predict(X_test_3d_a, verbose=0).flatten()
m = evaluate_model(y_test_a, preds, 'LSTM', 'Test')
results_anxiety.append(m)
print(f'[Anxiety] LSTM: MAE={m["MAE"]:.4f}, RMSE={m["RMSE"]:.4f}, R2={m["R2"]:.4f}')
lstm_a.save(str(ARTIFACT_DIR / 'anxiety_lstm.keras'))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history_c.history['loss'],     label='Train Loss', color='steelblue')
axes[0].plot(history_c.history['val_loss'], label='Val Loss',   color='coral')
axes[0].set_title('LSTM Training History - Craving')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].legend()

axes[1].plot(history_a.history['loss'],     label='Train Loss', color='steelblue')
axes[1].plot(history_a.history['val_loss'], label='Val Loss',   color='coral')
axes[1].set_title('LSTM Training History - Anxiety')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MSE Loss')
axes[1].legend()

plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'lstm_training_history.png', dpi=150, bbox_inches='tight')
plt.show()

## 15. Model Comparison and Evaluation Summary

All trained models are ranked by RMSE on the held-out test set. The comparison table provides a consolidated view to identify the best-performing algorithm for each target variable.

In [ ]:
results_df_c = pd.DataFrame(results_craving).sort_values('RMSE')
results_df_a = pd.DataFrame(results_anxiety).sort_values('RMSE')

print('=== Craving Score Forecast Results ===')
print(results_df_c.to_string(index=False))
print()
print('=== Anxiety Score Forecast Results ===')
print(results_df_a.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, rdf, title in zip(axes, [results_df_c, results_df_a], ['Craving Score', 'Anxiety Score']):
    colors = ['gold' if i == 0 else 'steelblue' for i in range(len(rdf))]
    bars   = ax.barh(rdf['Model'], rdf['RMSE'], color=colors, edgecolor='white')
    ax.set_xlabel('RMSE (lower is better)')
    ax.set_title(f'Model Comparison - {title}')
    ax.invert_yaxis()
    for bar, val in zip(bars, rdf['RMSE']):
        ax.text(val + 0.002, bar.get_y() + bar.get_height() / 2, f'{val:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
results_df_c.to_csv(ARTIFACT_DIR / 'results_craving.csv', index=False)
results_df_a.to_csv(ARTIFACT_DIR / 'results_anxiety.csv', index=False)
print('Result tables saved.')

## 16. Feature Importance Analysis

Feature importance is extracted from the best tree-based model (LightGBM) using the `gain` metric, which measures the average improvement in the loss function brought by each feature when it is used in a split. This analysis reveals which behavioral, physiological, and temporal features are most predictive of craving and anxiety dynamics.

In [ ]:
def plot_feature_importance(model, feature_names, top_n=25, title='Feature Importance', save_path=None):
    if hasattr(model, 'booster_'):
        importances = model.booster_.feature_importance(importance_type='gain')
    else:
        importances = model.feature_importances_

    fi_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
    fi_df = fi_df.sort_values('importance', ascending=False).head(top_n)

    fig, ax = plt.subplots(figsize=(10, top_n * 0.35 + 1))
    ax.barh(fi_df['feature'][::-1], fi_df['importance'][::-1], color='steelblue', edgecolor='white')
    ax.set_xlabel('Feature Importance (Gain)')
    ax.set_title(title)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    return fi_df

fi_c = plot_feature_importance(
    lgb_c, FEATURE_COLS,
    title='Top 25 Features - Craving Score (LightGBM)',
    save_path=ARTIFACT_DIR / 'feature_importance_craving.png'
)
fi_a = plot_feature_importance(
    lgb_a, FEATURE_COLS,
    title='Top 25 Features - Anxiety Score (LightGBM)',
    save_path=ARTIFACT_DIR / 'feature_importance_anxiety.png'
)

fi_c.to_csv(ARTIFACT_DIR / 'feature_importance_craving.csv', index=False)
fi_a.to_csv(ARTIFACT_DIR / 'feature_importance_anxiety.csv', index=False)
print('Feature importance saved.')

## 17. Residual Analysis

Residual analysis examines whether the model's prediction errors are randomly distributed (desirable) or exhibit systematic patterns (indicative of model misspecification). Plots include:

- **Residuals vs. Predicted**: Should show no funnel shape or trend.
- **Residual distribution**: Should be approximately normal and centered at zero.
- **Actual vs. Predicted scatter**: Should cluster tightly along the 45-degree diagonal.

In [ ]:
best_preds_c = lgb_c.predict(X_test_c_pp)
best_preds_a = lgb_a.predict(X_test_a_pp)
residuals_c  = y_test_c.values - best_preds_c
residuals_a  = y_test_a.values - best_preds_a

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for row_idx, (residuals, preds, y_true, label) in enumerate([
    (residuals_c, best_preds_c, y_test_c.values, 'Craving'),
    (residuals_a, best_preds_a, y_test_a.values, 'Anxiety')
]):
    axes[row_idx, 0].scatter(preds, residuals, alpha=0.3, s=8, color='steelblue')
    axes[row_idx, 0].axhline(0, color='red', linewidth=1, linestyle='--')
    axes[row_idx, 0].set_xlabel('Predicted')
    axes[row_idx, 0].set_ylabel('Residual')
    axes[row_idx, 0].set_title(f'{label} - Residuals vs. Predicted')

    axes[row_idx, 1].hist(residuals, bins=50, color='coral', edgecolor='white')
    axes[row_idx, 1].axvline(0, color='red', linewidth=1, linestyle='--')
    axes[row_idx, 1].set_xlabel('Residual')
    axes[row_idx, 1].set_ylabel('Frequency')
    axes[row_idx, 1].set_title(f'{label} - Residual Distribution')

    axes[row_idx, 2].scatter(y_true, preds, alpha=0.3, s=8, color='mediumseagreen')
    lims = [min(y_true.min(), preds.min()), max(y_true.max(), preds.max())]
    axes[row_idx, 2].plot(lims, lims, 'r--', linewidth=1)
    axes[row_idx, 2].set_xlabel('Actual')
    axes[row_idx, 2].set_ylabel('Predicted')
    axes[row_idx, 2].set_title(f'{label} - Actual vs. Predicted')

plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'residual_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 18. Forecast Visualization by Day

The population-level forecast is visualized by aggregating predictions and actuals across all participants for each test day (days 26-30). This plot conveys the model's ability to track day-over-day fluctuations at the aggregate level.

In [ ]:
test_df_full_c = df[test_mask].dropna(subset=[TARGET_CRAVING]).copy()
test_df_full_a = df[test_mask].dropna(subset=[TARGET_ANXIETY]).copy()

test_df_full_c['pred'] = best_preds_c
test_df_full_a['pred'] = best_preds_a

agg_c = test_df_full_c.groupby(TIME_COL).agg(
    actual=(TARGET_CRAVING, 'mean'), predicted=('pred', 'mean')
).reset_index()
agg_a = test_df_full_a.groupby(TIME_COL).agg(
    actual=(TARGET_ANXIETY, 'mean'), predicted=('pred', 'mean')
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, agg_df, title in zip(axes, [agg_c, agg_a], ['Overall Craving Score', 'Daily Anxiety Score']):
    ax.plot(agg_df[TIME_COL], agg_df['actual'],    'o-',  label='Actual',             color='steelblue', linewidth=2)
    ax.plot(agg_df[TIME_COL], agg_df['predicted'], 's--', label='Predicted (LightGBM)', color='coral',     linewidth=2)
    ax.set_xlabel('Day Number')
    ax.set_ylabel('Mean Score')
    ax.set_title(f'Test Set Forecast - {title}')
    ax.legend()

plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'forecast_visualization.png', dpi=150, bbox_inches='tight')
plt.show()

## 19. Per-Participant Forecast

To validate that the model generalizes at the individual level rather than only in aggregate, forecasts are plotted for a sample of participants across the test horizon.

In [ ]:
sample_pids = test_df_full_c[GROUP_COL].unique()[:4]
fig, axes   = plt.subplots(2, 2, figsize=(16, 10))

for ax, pid in zip(axes.flatten(), sample_pids):
    sub = test_df_full_c[test_df_full_c[GROUP_COL] == pid].sort_values(TIME_COL)
    ax.plot(sub[TIME_COL], sub[TARGET_CRAVING], 'o-',  label='Actual Craving',    color='steelblue', linewidth=2)
    ax.plot(sub[TIME_COL], sub['pred'],         's--', label='Predicted Craving', color='coral',     linewidth=2)
    ax.set_title(f'Participant {pid} - Craving Forecast (Days 26-30)')
    ax.set_xlabel('Day Number')
    ax.set_ylabel('Overall Craving Score')
    ax.legend(fontsize=9)

plt.suptitle('Individual-Level Craving Forecast (LightGBM)', fontsize=13)
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'per_participant_forecast.png', dpi=150, bbox_inches='tight')
plt.show()

## 20. Cross-Validation with TimeSeriesSplit

`TimeSeriesSplit` provides a rigorous, forward-chaining cross-validation strategy that respects the temporal order of observations. Unlike standard k-fold, no future data leaks into the training fold at any split. The best model (LightGBM) is evaluated across 5 folds.

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)

df_cv = df.dropna(subset=[TARGET_CRAVING]).sort_values([GROUP_COL, TIME_COL])
X_cv  = preprocessor.transform(df_cv[FEATURE_COLS])
y_cv  = df_cv[TARGET_CRAVING]

cv_lgb = lgb.LGBMRegressor(
    n_estimators=300, learning_rate=0.05, max_depth=6,
    num_leaves=63, subsample=0.8, colsample_bytree=0.8,
    random_state=SEED, n_jobs=-1, verbose=-1
)

cv_scores   = cross_val_score(cv_lgb, X_cv, y_cv, cv=tscv,
                              scoring='neg_root_mean_squared_error', n_jobs=-1)
rmse_scores = -cv_scores

print(f'TimeSeriesSplit CV RMSE scores (Craving): {np.round(rmse_scores, 4)}')
print(f'Mean RMSE : {rmse_scores.mean():.4f}')
print(f'Std  RMSE : {rmse_scores.std():.4f}')

## 21. Artifact Export and Experiment Summary

A complete experiment summary is serialized to a JSON file. This metadata file records the best model for each target, key performance metrics, feature engineering decisions, and a timestamp, enabling full reproducibility and audit traceability.

In [ ]:
best_craving_row = results_df_c.iloc[0].to_dict()
best_anxiety_row = results_df_a.iloc[0].to_dict()

summary = {
    'experiment': 'Nicotine Craving and Anxiety Time-Series Forecasting',
    'timestamp': datetime.now().isoformat(),
    'dataset': {
        'path': str(DATA_PATH),
        'total_rows': int(len(df_raw)),
        'participants': int(df_raw[GROUP_COL].nunique()),
        'days_per_participant': int(df_raw[TIME_COL].max())
    },
    'targets': [TARGET_CRAVING, TARGET_ANXIETY],
    'temporal_split': {
        'train_days': f'1-{TRAIN_END}',
        'val_days':   f'{TRAIN_END+1}-{VAL_END}',
        'test_days':  f'{VAL_END+1}-30'
    },
    'feature_engineering': {
        'lag_periods':   LAG_PERIODS,
        'roll_windows':  ROLL_WINDOWS,
        'total_features': len(FEATURE_COLS)
    },
    'models_trained': [r['Model'] for r in results_craving],
    'best_craving_model': best_craving_row,
    'best_anxiety_model': best_anxiety_row,
    'cross_validation': {
        'strategy': 'TimeSeriesSplit',
        'n_splits': 5,
        'craving_rmse_mean': round(float(rmse_scores.mean()), 4),
        'craving_rmse_std':  round(float(rmse_scores.std()),  4)
    }
}

with open(ARTIFACT_DIR / 'experiment_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('Experiment summary saved.')
print(json.dumps(summary, indent=2))

In [ ]:
feature_metadata = {
    'feature_columns': FEATURE_COLS,
    'target_craving': TARGET_CRAVING,
    'target_anxiety': TARGET_ANXIETY,
    'group_col': GROUP_COL,
    'time_col': TIME_COL,
    'label_encoded_columns': categorical_cols,
    'lag_features':       [f for f in FEATURE_COLS if 'lag'  in f],
    'rolling_features':   [f for f in FEATURE_COLS if 'roll' in f],
    'engineered_features': [f for f in FEATURE_COLS
                            if any(k in f for k in ['Ratio', 'Composite', 'x_', 'Minutes'])]
}

with open(ARTIFACT_DIR / 'feature_metadata.json', 'w') as f:
    json.dump(feature_metadata, f, indent=2)

print(f'Feature metadata saved. Total features: {len(FEATURE_COLS)}')

## 22. Artifacts Summary

All model artifacts and analysis outputs have been saved to the `Model/artifacts/` directory.

| Artifact | Description |
|---|---|
| `preprocessor_craving.pkl` | Fitted imputer + scaler pipeline for craving |
| `preprocessor_anxiety.pkl` | Fitted imputer + scaler pipeline for anxiety |
| `craving_lightgbm.txt` | Best LightGBM model for craving (native format) |
| `anxiety_lightgbm.txt` | Best LightGBM model for anxiety (native format) |
| `craving_xgboost.json` | XGBoost model for craving |
| `anxiety_xgboost.json` | XGBoost model for anxiety |
| `craving_lstm.keras` | LSTM deep learning model for craving |
| `anxiety_lstm.keras` | LSTM deep learning model for anxiety |
| `results_craving.csv` | Full evaluation results table for craving |
| `results_anxiety.csv` | Full evaluation results table for anxiety |
| `feature_importance_craving.csv` | Feature gain importances for craving |
| `feature_importance_anxiety.csv` | Feature gain importances for anxiety |
| `experiment_summary.json` | Complete experiment metadata and results |
| `feature_metadata.json` | Column schema and feature group registry |
| `target_distributions.png` | Histograms of target variables |
| `temporal_trends.png` | Population-level temporal trends |
| `individual_trajectories.png` | Six sample participant trajectories |
| `correlation_heatmap.png` | Feature correlation matrix |
| `model_comparison.png` | RMSE bar chart across all models |
| `lstm_training_history.png` | LSTM epoch loss curves |
| `residual_analysis.png` | Residual diagnostic plots |
| `forecast_visualization.png` | Aggregate actual vs. predicted on test set |
| `per_participant_forecast.png` | Individual-level forecast for 4 participants |

In [ ]:
print('All artifacts saved in:', ARTIFACT_DIR.resolve())
for p in sorted(ARTIFACT_DIR.iterdir()):
    size_kb = p.stat().st_size / 1024
    print(f'  {p.name:<45} {size_kb:>8.1f} KB')